In [13]:
import pandas as pd
import glob
import os
import numpy as np
import shutil
from dictionaries import *

In [14]:
## MID
dir = "..\\data\\granger\\26Q3\\MID"
field_sdg = "4065.04"
task_code = "MID_3Q26"
task_code2 = "MID_Quarterly"

## Grand River
# dir = "..\\data\\granger\\26Q3\\Grand River"
# field_sdg = "4065.02"
# task_code = "GR_3Q26"
# task_code2 = "GR_Quarterly"

## Wood Street
# dir = "..\\data\\granger\\26Q3\\Wood Street"
# field_sdg = ""
# task_code = "WS_3Q26"
# task_code2 = ""

# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
print(excel_paths)

idx = 0  # Change this index to select a different file

EDD_NAME = excel_paths[idx].split("\\")[-1]
EDD_TO_PROCESS = pd.read_excel(f'{dir}\\{EDD_NAME}')
lab_sgd = EDD_NAME.split(" ")[0]
print(lab_sgd)
EDD_TO_PROCESS.shape

['..\\data\\granger\\26Q3\\MID\\26G0035 FINAL granger 17 Jul 26 1222.xlsx', '..\\data\\granger\\26Q3\\MID\\26G0047 FINAL granger 17 Jul 26 1226.xlsx', '..\\data\\granger\\26Q3\\MID\\26G0067 FINAL granger 17 Jul 26 1412.xlsx', '..\\data\\granger\\26Q3\\MID\\26G0092 FINAL granger 17 Jul 26 1234.xlsx', '..\\data\\granger\\26Q3\\MID\\26G0113 FINAL granger 17 Jul 26 1251.xlsx']
26G0035


(221, 26)

In [15]:
template_path = r"..\EDDTemplate\ESBasic_TRC Format.xlsx"
template_df = pd.read_excel(template_path, sheet_name="ESBasic_TRC")

In [16]:
EDD_TO_PROCESS['Sample_ID'].unique()

<StringArray>
['PW-50', 'PW-49', 'PW-46', 'PW-48', 'Leachate']
Length: 5, dtype: str

In [17]:
expected_columns = ['Lab_ID', 'Sample_ID', 'Analyte', 'Prefix', 'Result', 'Concentration', 'Comment', 'AnalyticalSummary', 'Units', 'Matrix', 'Sample_Collection_Date', 'Laboratory_Received_Date', 'Date_Extracted', 'Date_Analyzed', 'Laboratory_Reporting_Date', 'Detection_Limits', 'Reporting_Limits', 'Dilution_Factor', 'CAS Number', 'Extraction_Method', 'Sample_Fraction', 'Analytical_Method_Description', 'Analytical_Method_Reference', 'Laboratory_QC_Level', 'Project_Name', 'Project_Number']
try:
    assert EDD_TO_PROCESS.columns.tolist() == expected_columns
    print("Column names match the expected list.")
except AssertionError:
    print("Column names do not match the expected list.")
    # The EDD to process has extra columns that are not in the expected list. Print the extra columns.
    print("Extra columns in EDD to process:", set(EDD_TO_PROCESS.columns.tolist()) - set(expected_columns))

Column names match the expected list.


In [18]:
EDD_TO_PROCESS.Analytical_Method_Reference.unique()

<StringArray>
[        'EPA 6020B',         'EPA 200.8',         'EPA 624.1',
         'EPA 9056A',         'Hach 8000',    'SM 2320 B-2021',
    'SM 2510 B-2011',     'SM 4500 NH3 G', 'SM 4500-H+ B 2011',
     'SM 5310B-2014',     'ASTM D7511-12',         'EPA 245.1',
         'EPA 420.1',         'EPA 8260D',     'SM 5210B-2016']
Length: 15, dtype: str

In [ ]:
EDD = EDD_TO_PROCESS.copy()
EDD.rename(columns=edd_to_template_mapping, inplace=True)

# Blank result_value when a less-than indicator is present
mask_lt = EDD['result_value'].astype(str).str.contains('<', na=False)
# Blank result_value when a less-than indicator is present
mask_lt = EDD['Prefix'].astype(str).str.strip().eq('<')
EDD.loc[mask_lt, 'result_value'] = np.nan

# remove leading and trailing whitespace from 'result_value' and convert to string
EDD['result_value'] = EDD['result_value'].fillna('').astype(str).str.strip()

# 3. Where 'Result' is empty, set 'detect_flag' to "N" otherwise set it to "Y"
EDD['detect_flag'] = np.where(EDD['result_value'] == '', "N", "Y")

# 1. Handle Datetimes safely
temp_datetime = pd.to_datetime(EDD_TO_PROCESS["Sample_Collection_Date"], errors='coerce')
EDD["sample_date"] = temp_datetime.dt.strftime("%m/%d/%Y")  # 04/13/2026
EDD["sample_time"] = temp_datetime.dt.strftime("%H:%M:%S")  # 09:10:00

# 2. Preserve sample_name BEFORE modifying sys_loc_code, then assign matrix
EDD['sample_name'] = EDD['sys_loc_code']

# 3. Fill in Cas RN where the dictionary has a match, otherwise leave it as is (which may be NaN or a previously filled value)
EDD['cas_rn'] = EDD['chemical_name'].map(analytes_cas_dict).fillna(EDD_TO_PROCESS['CAS Number'])

# mappings
EDD['analytic_method'] = EDD['analytic_method'].map(edd_to_db_methods)
EDD['prep_method'] = EDD['prep_method'].map(edd_to_db_Prepmethods)


EDD['result_unit'] = EDD['result_unit'].map(result_unit_dict).fillna(EDD_TO_PROCESS['Units'])  # Map to standardized units if possible, else keep original code
EDD['detection_limit_unit'] = EDD['result_unit']
EDD.loc[EDD['reporting_detection_limit'].isna(), 'detection_limit_unit'] = None
mask = EDD['result_value'].isna() | (EDD['result_value'].astype(str).str.strip() == "")
EDD.loc[mask, 'result_unit'] = None


# Map to standardized sample names if possible, else keep original code
EDD['sys_loc_code'] = EDD['sys_loc_code'].map(sample_id_dict).fillna(EDD['sys_loc_code'])  
EDD['sample_matrix_code'] = EDD['sys_loc_code'].apply(assign_matrix)

# Map to lab_matrix if possible, else keep original code
EDD['lab_matrix'] = EDD['lab_matrix'].map(lab_matrix_dict).fillna(EDD_TO_PROCESS['Matrix'])  

# 4. Generate #sys_sample_code safely
date_yyyymmdd = pd.to_datetime(EDD["sample_date"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
EDD['#sys_sample_code'] = EDD['sample_name'].str.strip() + "_" + date_yyyymmdd

# 6. FIX: Use np.select to evaluate sample types conditionally without overwriting
sample_name_lower = EDD["sample_name"].astype(str).str.lower()
conditions = [
    sample_name_lower.str.contains("equipment", na=False),
    sample_name_lower.str.contains("trip", na=False),
    sample_name_lower.str.contains("duplicate", na=False) | sample_name_lower.str.contains("dupa", na=False)| sample_name_lower.str.contains("dupb", na=False)| sample_name_lower.str.contains("dupc", na=False),
    sample_name_lower.str.contains("field", na=False)
]
choices = ["EB", "TB", "FD", "FB"]
EDD['sample_type_code'] = np.select(conditions, choices, default="N")

# where the sample_type_code is "EB", "TB", "or "FB" clear out the sys_loc_code  
EDD.loc[EDD['sample_type_code'].isin(["EB", "TB", "FB", "FD"]), 'sys_loc_code'] = None

# Fractions
EDD['fraction'] = "N"  # Default fraction to "N"
# if EDD_TO_PROCESS['Extraction_Method'] contains "Metals Filter" or "Dissolved", set fraction to "D"
EDD.loc[EDD_TO_PROCESS['Extraction_Method'].str.contains("EPA 200.2", case=False, na=False), 'fraction'] = "T"
EDD.loc[EDD_TO_PROCESS['Extraction_Method'].str.contains("Metals Filter", case=False, na=False), 'fraction'] = "D"
EDD.loc[EDD_TO_PROCESS['Extraction_Method'].str.contains("Dissolved", case=False, na=False), 'fraction'] = "D"
EDD.loc[EDD['chemical_name'].str.contains("dissolved", case=False, na=False), 'fraction'] = "D"
EDD.loc[EDD['chemical_name'].str.contains("total", case=False, na=False), 'fraction'] = "T"
EDD.loc[EDD['chemical_name'].str.contains("Total Organic Carbon", case=False, na=False), 'fraction'] = "N"
EDD.loc[EDD['chemical_name'].str.contains("Total Inorganic Nitrogen", case=False, na=False), 'fraction'] = "N"
EDD.loc[EDD['chemical_name'].str.contains("Total Dissolved Solids", case=False, na=False), 'fraction'] = "N"
EDD.loc[EDD['chemical_name'].str.contains("Total Phenolics", case=False, na=False), 'fraction'] = "N"
EDD.loc[EDD['chemical_name'].str.contains("Xylenes, total", case=False, na=False), 'fraction'] = "N"
EDD.loc[EDD['chemical_name'].str.contains("Cyanide", case=False, na=False), 'fraction'] = "N"

# In one line, fix the dilution factor column to not exceed 1 decimal place, but also not convert integers to floats unnecessarily
EDD['dilution_factor'] = EDD['dilution_factor'].round(1).where(EDD['dilution_factor'].notnull(), None)

# Final Fill-Ins for Required Fields
EDD['result_type_code'] = "TRG"
EDD['reportable_result'] = "Yes"
EDD['test_type'] = "Initial"
EDD['Lab_SDG'] = lab_sgd
# EDD['prep_method'] = "Method"
# 8. FIX: Reindex at the very end to ensure temporary calculation spaces aren't mutated prematurely
EDD = EDD.reindex(columns=template_df.columns)


EDD['Field_SDG'] = field_sdg
EDD['task_code'] = task_code
EDD['task_code_2'] = task_code2

In [20]:
# Show any unknowing TICs for review
EDD[EDD.analytic_method.isna()][['#sys_sample_code','sys_loc_code', 'chemical_name', 'cas_rn', 'analytic_method']]

,#sys_sample_code,sys_loc_code,chemical_name,cas_rn,analytic_method


In [21]:
# Show any unknowing TICs for review
EDD[EDD.cas_rn.str.contains('Unknown TIC')][['#sys_sample_code','sys_loc_code', 'chemical_name', 'reporting_detection_limit', 'detection_limit_unit', 'parent_sample_code', 'cas_rn']]

,#sys_sample_code,sys_loc_code,chemical_name,reporting_detection_limit,detection_limit_unit,parent_sample_code,cas_rn


In [22]:
EDD.head()

,#sys_sample_code,sample_name,sample_type_code,sample_matrix_code,sample_date,sample_time,sys_loc_code,parent_sample_code,start_depth,end_depth,...,workflow_status,task_code_2,TRC_Reason_Codes,QCI,QCII,QCIII,Lab_Cert_ID_No,AltParameterCode,Approval_Code,equipment_code
0,PW-50_20260706,PW-50,N,WG,07/06/2026,00:00:00,PW-50,NaN,NaN,NaN,...,NaN,MID_Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PW-49_20260706,PW-49,N,WG,07/06/2026,00:00:00,PW-49,NaN,NaN,NaN,...,NaN,MID_Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PW-46_20260706,PW-46,N,WG,07/06/2026,00:00:00,PW-46,NaN,NaN,NaN,...,NaN,MID_Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PW-46_20260706,PW-46,N,WG,07/06/2026,00:00:00,PW-46,NaN,NaN,NaN,...,NaN,MID_Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PW-46_20260706,PW-46,N,WG,07/06/2026,00:00:00,PW-46,NaN,NaN,NaN,...,NaN,MID_Quarterly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{EDD_NAME.split('.')[0]}_ESBasic_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    EDD.to_excel(
        writer,
        sheet_name="ESBasic_TRC",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

In [24]:
## Mapping script to find the best matches between your EDD methods and the database codes, and print a dictionary for you to copy/paste into your main script. This is a helper script to make the mapping process easier, especially when method names don't match exactly but are close enough to be recognizable.
import pandas as pd
import re

# 1. Your new EDD methods
edd_methods = [
    'EPA 200.8 Rev. 5.4', 'EPA 200.7 Rev. 4.4', 'EPA 8260D', 'Calculation', 
    'EPA 350.1 Rev. 2.0', 'SM 2540 C-20', 'SM 2320 B-21', 'SM 5310 B-14', 
    'EPA 300.0 Rev. 2.1', 'SM 4500-Cl D-21', 'EPA 410.4 Rev. 2.0'
]

# 2. Load your reference file and grab the first column
try:
    df_ref = pd.read_excel('rt_analytic_method.xlsx')
    db_codes = df_ref.iloc[:, 0].dropna().astype(str).unique().tolist()
except FileNotFoundError:
    print("Error: Could not find the file. Make sure the script and CSV are in the same folder.")
    db_codes = []

# 3. Helper function to strip out spaces, punctuation, and "Rev" versions for matching
def normalize(text):
    text = str(text).upper().replace(" ", "")
    text = re.sub(r'REV\..*$', '', text) # Strip "Rev. X.X"
    text = re.sub(r'[^A-Z0-9]', '', text) # Keep only alphanumeric
    return text

method_mapping = {}

# 4. Find the best match
if db_codes:
    for edd_m in edd_methods:
        norm_edd = normalize(edd_m)
        matched = False
        
        for db_c in db_codes:
            norm_db = normalize(db_c)
            # Check if the core letters/numbers match
            if norm_edd in norm_db or norm_db in norm_edd:
                method_mapping[edd_m] = db_c
                matched = True
                break
                
        # Flag any methods that didn't find a clean match
        if not matched:
            method_mapping[edd_m] = "MANUAL_CHECK_REQUIRED"

    # 5. Print the beautifully formatted dictionary
    print("Here is your dictionary:\n")
    print("edd_to_db_methods = {")
    for key, value in method_mapping.items():
        print(f"    '{key}': '{value}',")
    print("}")

Error: Could not find the file. Make sure the script and CSV are in the same folder.
